# Pre-generate capstone scenarios on Kaggle

Runs the evaluated model recipe over a bundle from `python -m app.scripts.export_records`
and writes a JSONL file for `python -m app.scripts.import_scenarios`.

Before running:

1. Upload the bundle folder (`bundle.json`, `scenario_contract.py`, `model_runtime.py`,
   `requirements.txt`) as a private Kaggle dataset and add it to this notebook.
2. Add a Kaggle secret named `HF_TOKEN` (Add-ons -> Secrets) that can read the private
   adapter repo. The base model is not gated and needs no token. The notebook never prints it.
3. Choose a GPU accelerator (T4). Kaggle quota and availability are checked when you run,
   not assumed.

The prompt format (`structured-p1` or `training`) is the one chosen when the bundle was
exported with `--prompt-format`; it must match the app's `SCENARIO_PROMPT_FORMAT` for the
imported entries to be found by new requests.

The code that loads and samples the model is the bundle's copy of `model_runtime.py`, the
same file the Space runs, checked against the hashes in `bundle.json`.

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

# The bundle dataset: the folder under /kaggle/input that holds bundle.json.
BUNDLE_DIR = next(path.parent for path in Path("/kaggle/input").rglob("bundle.json"))
OUTPUT = Path("/kaggle/working/scenarios.jsonl")

bundle = json.loads((BUNDLE_DIR / "bundle.json").read_text(encoding="utf-8"))
for name, expected in bundle["files"].items():
    actual = hashlib.sha256((BUNDLE_DIR / name).read_bytes()).hexdigest()
    assert actual == expected, f"{name} does not match bundle.json; re-export the bundle"
print(f"Bundle {BUNDLE_DIR}: analysis {bundle['analysis_id']}, {len(bundle['items'])} path(s)")

Model provenance. The bundle carries it when `export_records` was given `--base-revision`,
`--adapter-repo` and `--adapter-revision`; otherwise fill in the full commit SHAs here.
They must be the revisions the app will be configured with, or the imported entries will
never match a live request.

In [ ]:
PINNED = bundle.get("provenance") or {}
BASE_REVISION = PINNED.get("model_revision") or ""      # 40-character commit SHA
ADAPTER_REPO = PINNED.get("adapter_repo") or ""         # OWNER/NAME
ADAPTER_REVISION = PINNED.get("adapter_revision") or "" # 40-character commit SHA
LOADING_RECIPE = PINNED.get("loading_recipe") or "nf4-dq-unmerged"

Install the Space's pinned libraries, except torch (Kaggle's CUDA build stays) and
`spaces` (ZeroGPU only). The torch version actually used is recorded in every result.

In [ ]:
import subprocess

wanted = [
    line.strip() for line in (BUNDLE_DIR / "requirements.txt").read_text().splitlines()
    if line.strip() and not line.startswith("#") and line.split("==")[0].strip() not in {"torch", "spaces"}
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *wanted], check=True)

In [ ]:
sys.path.insert(0, str(BUNDLE_DIR))
import scenario_contract as contract
import model_runtime

# The bundle's prompts were built in one format (export_records --prompt-format);
# bundles from before the format was named are the default structured prompt.
PROMPT_FORMAT = bundle.get("prompt_format", contract.DEFAULT_PROMPT_FORMAT)
PROMPT_VERSION = contract.prompt_version_for(PROMPT_FORMAT)
assert bundle["prompt_version"] == PROMPT_VERSION, "bundle and contract prompt versions differ"
print(f"Prompt format {PROMPT_FORMAT}, version {PROMPT_VERSION}")
contract.make_provenance(
    model_revision=BASE_REVISION,
    adapter_repo=ADAPTER_REPO,
    adapter_revision=ADAPTER_REVISION,
    loading_recipe=LOADING_RECIPE,
)

In [ ]:
from kaggle_secrets import UserSecretsClient

loaded = model_runtime.load(
    model_revision=BASE_REVISION,
    adapter_repo=ADAPTER_REPO,
    adapter_revision=ADAPTER_REVISION,
    loading_recipe=LOADING_RECIPE,
    token=UserSecretsClient().get_secret("HF_TOKEN"),
)
print(json.dumps(model_runtime.metadata(loaded), indent=2))

Generate. Results are appended one line at a time, so a rerun after an interruption skips
the paths already written.

In [ ]:
done = set()
if OUTPUT.exists():
    done = {json.loads(line)["entry"]["cache_key"] for line in OUTPUT.read_text().splitlines() if line.strip()}

provenance = loaded.provenance(contract.MAX_NEW_TOKENS)
with OUTPUT.open("a", encoding="utf-8") as out:
    for item in bundle["items"]:
        key = contract.cache_key(item["prompt"], provenance, PROMPT_VERSION)
        if key in done:
            print(f"{item['source_id']}: already written")
            continue
        prompt_ids = model_runtime.encode_prompt(loaded, item["prompt"], contract.MAX_NEW_TOKENS)
        response = model_runtime.generate(loaded, prompt_ids, contract.MAX_NEW_TOKENS)
        contract.validate_result(response, provenance)
        entry = contract.make_cache_entry(
            prompt=item["prompt"], provenance=provenance, response=response, source="pregenerated",
            prompt_version=PROMPT_VERSION)
        line = contract.make_result_line(
            analysis_id=bundle["analysis_id"], source_id=item["source_id"], record=item["record"], entry=entry)
        out.write(contract.canonical_json(line) + "\n")
        out.flush()
        sections = contract.parse_sections(response["text"])
        print(f"{item['source_id']}: {response['new_tokens']} tokens in {response['seconds']:.1f}s, "
              f"finish {response['finish_reason']}, missing {sections['missing'] or 'none'}")
print(f"Wrote {OUTPUT}")

Download `/kaggle/working/scenarios.jsonl` and import it from the repository root:

```
python -m app.scripts.import_scenarios scenarios.jsonl
```